# SupplyShield AI — SQL Database Integration

This notebook converts the final risk-scored intelligence generated by the
SupplyShield AI pipeline into a normalized relational SQL database.

Pipeline:

Risk-Scored Dataset
→ Data Validation
→ Relational Transformation
→ SQL Schema
→ Data Ingestion
→ Integrity Validation
→ Analytical Queries
→ Dashboard-Ready Views

In [1]:
# ============================================================
# SUPPLYSHIELD AI
# CELL 2 — IMPORTS & DATABASE CONFIGURATION
# ============================================================

from __future__ import annotations

import json
import math
import re
import sqlite3
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 180)


# ============================================================
# PROJECT PATH CONFIGURATION
# ============================================================

PROJECT_ROOT = Path.cwd()

# Locate repository root
for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):

    if (parent / "member2").exists():

        PROJECT_ROOT = parent
        break


PROCESSED_DIR = (
    PROJECT_ROOT
    / "member2"
    / "data"
    / "processed"
)


RISK_OUTPUT_DIR = (
    PROCESSED_DIR
    / "risk_scoring"
)


DATABASE_DIR = (
    PROJECT_ROOT
    / "member2"
    / "database"
)


DATABASE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


DATABASE_PATH = (
    DATABASE_DIR
    / "supplyshield.db"
)


FINAL_JSON = (
    RISK_OUTPUT_DIR
    / "final_risk_scored_supply_data.json"
)


FINAL_CSV = (
    RISK_OUTPUT_DIR
    / "final_risk_scored_supply_data.csv"
)


print("=" * 80)
print("SUPPLYSHIELD AI — SQL DATABASE INTEGRATION")
print("=" * 80)

print(f"Project root : {PROJECT_ROOT}")
print(f"Input JSON   : {FINAL_JSON}")
print(f"Input CSV    : {FINAL_CSV}")
print(f"Database     : {DATABASE_PATH}")

print("=" * 80)

SUPPLYSHIELD AI — SQL DATABASE INTEGRATION
Project root : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI
Input JSON   : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\data\processed\risk_scoring\final_risk_scored_supply_data.json
Input CSV    : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\data\processed\risk_scoring\final_risk_scored_supply_data.csv
Database     : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\database\supplyshield.db


## 1. Load the Final Risk Intelligence

The database layer consumes the validated output of Notebook 05.
The JSON dataset is used as the canonical source while CSV remains available
for interoperability and manual inspection.

In [2]:
# ============================================================
# CELL 4 — LOAD NOTEBOOK 05 OUTPUT
# ============================================================

def load_final_dataset():

    if FINAL_JSON.exists():

        print("Loading canonical JSON dataset...")

        with open(
            FINAL_JSON,
            "r",
            encoding="utf-8"
        ) as file:

            records = json.load(file)

        dataframe = pd.DataFrame(records)

        source = FINAL_JSON

    elif FINAL_CSV.exists():

        print(
            "JSON not found. Falling back to CSV..."
        )

        dataframe = pd.read_csv(
            FINAL_CSV
        )

        source = FINAL_CSV

    else:

        raise FileNotFoundError(
            "\nFinal risk-scored dataset not found.\n\n"
            "Expected one of:\n"
            f"{FINAL_JSON}\n"
            f"{FINAL_CSV}\n\n"
            "Run Notebook 05 first."
        )

    return dataframe, source


df, input_source = load_final_dataset()


# ============================================================
# BASIC VALIDATION
# ============================================================

if df.empty:

    raise ValueError(
        "Final risk dataset is empty."
    )


print("=" * 80)
print("FINAL RISK DATASET LOADED")
print("=" * 80)

print(f"Source       : {input_source}")
print(f"Records      : {len(df):,}")
print(f"Columns      : {len(df.columns):,}")

print("\nRisk columns detected:")

risk_columns = [
    column
    for column in df.columns
    if "risk" in str(column).lower()
]

for column in risk_columns:

    print(f"  ✓ {column}")


print("=" * 80)

Loading canonical JSON dataset...
FINAL RISK DATASET LOADED
Source       : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\data\processed\risk_scoring\final_risk_scored_supply_data.json
Records      : 303
Columns      : 168

Risk columns detected:
  ✓ preliminary_risk_band
  ✓ availability_risk_score
  ✓ rating_risk_score
  ✓ text_risk_score
  ✓ supplier_mean_availability_risk
  ✓ supplier_mean_text_risk
  ✓ supplier_data_quality_risk
  ✓ source_mean_availability_risk
  ✓ source_mean_text_risk
  ✓ preliminary_risk_signal
  ✓ preliminary_risk_score
  ✓ nlp_supplier_risk_count
  ✓ nlp_quality_risk_count
  ✓ nlp_counterfeit_risk_count
  ✓ nlp_primary_risk_category
  ✓ nlp_supplier_risk_score
  ✓ nlp_quality_risk_score
  ✓ nlp_counterfeit_risk_score
  ✓ nlp_risk_score
  ✓ nlp_risk_score_100
  ✓ nlp_risk_band
  ✓ nlp_risk_reasons
  ✓ nlp_supplier_risk_categories
  ✓ nlp_supplier_risk_terms
  ✓ nlp_supplier_risk_score_100
  ✓ nlp_supplier_risk_band
  ✓ nlp_supplier_risk_re

In [3]:
# ============================================================
# CELL 5 — PRE-DATABASE DATA QUALITY VALIDATION
# ============================================================

print("=" * 80)
print("PRE-DATABASE DATA QUALITY AUDIT")
print("=" * 80)


quality_report = pd.DataFrame({
    "column": df.columns,
    "dtype": [
        str(df[column].dtype)
        for column in df.columns
    ],
    "missing_count": [
        int(df[column].isna().sum())
        for column in df.columns
    ],
    "missing_percentage": [
        round(
            float(df[column].isna().mean() * 100),
            2
        )
        for column in df.columns
    ],
    "unique_values": [
        int(df[column].nunique(dropna=True))
        for column in df.columns
    ]
})


print(
    f"Duplicate rows: {df.duplicated().sum():,}"
)

print(
    f"Total missing cells: "
    f"{df.isna().sum().sum():,}"
)

print(
    f"Memory usage: "
    f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)


display(
    quality_report
    .sort_values(
        "missing_percentage",
        ascending=False
    )
    .head(30)
)

PRE-DATABASE DATA QUALITY AUDIT
Duplicate rows: 0
Total missing cells: 48
Memory usage: 1.25 MB


,column,dtype,missing_count,missing_percentage,unique_values
8,currency,object,48,15.84,2
0,record_id,object,0,0.00,303
2,title,object,0,0.00,302
1,source,object,0,0.00,3
4,supplier,object,0,0.00,1
5,product,object,0,0.00,288
6,event,object,0,0.00,1
3,company,object,0,0.00,273
7,location,object,0,0.00,1
9,url,object,0,0.00,303


## 2. Relational Database Architecture

The denormalized risk-scored dataset is transformed into multiple relational
tables.

Core tables:

- `suppliers`
- `products`
- `supply_events`
- `risk_scores`
- `risk_signals`
- `pipeline_runs`

This structure allows the dashboard and future WebShield layer to query
specific business entities instead of repeatedly processing the complete
raw dataset.

In [4]:
# ============================================================
# CELL 7 — DATABASE CONNECTION
# ============================================================

connection = sqlite3.connect(
    DATABASE_PATH
)

connection.execute(
    "PRAGMA foreign_keys = ON;"
)

connection.execute(
    "PRAGMA journal_mode = WAL;"
)

connection.execute(
    "PRAGMA synchronous = NORMAL;"
)

print("=" * 80)
print("SQLITE DATABASE CONNECTED")
print("=" * 80)

print(
    f"Database file : {DATABASE_PATH}"
)

print(
    f"SQLite version: "
    f"{sqlite3.sqlite_version}"
)

print("=" * 80)

SQLITE DATABASE CONNECTED
Database file : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\database\supplyshield.db
SQLite version: 3.45.3


In [5]:
# ============================================================
# CELL 8 — RELATIONAL DATABASE SCHEMA
# ============================================================

cursor = connection.cursor()


# ============================================================
# SUPPLIERS
# ============================================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS suppliers (

    supplier_id INTEGER PRIMARY KEY AUTOINCREMENT,

    supplier_name TEXT NOT NULL,

    company_name TEXT,

    source TEXT,

    location TEXT,

    created_at TEXT NOT NULL,

    UNIQUE (
        supplier_name,
        company_name
    )
);
""")


# ============================================================
# PRODUCTS
# ============================================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS products (

    product_id INTEGER PRIMARY KEY AUTOINCREMENT,

    product_name TEXT NOT NULL,

    company_name TEXT,

    category TEXT,

    source TEXT,

    created_at TEXT NOT NULL,

    UNIQUE (
        product_name,
        company_name,
        source
    )
);
""")


# ============================================================
# SUPPLY EVENTS
# ============================================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS supply_events (

    event_id INTEGER PRIMARY KEY AUTOINCREMENT,

    supplier_id INTEGER,

    product_id INTEGER,

    event_type TEXT,

    event_title TEXT,

    event_description TEXT,

    location TEXT,

    source TEXT,

    event_date TEXT,

    record_hash TEXT UNIQUE,

    created_at TEXT NOT NULL,

    FOREIGN KEY (
        supplier_id
    )
    REFERENCES suppliers(supplier_id),

    FOREIGN KEY (
        product_id
    )
    REFERENCES products(product_id)
);
""")


# ============================================================
# RISK SCORES
# ============================================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS risk_scores (

    risk_id INTEGER PRIMARY KEY AUTOINCREMENT,

    supplier_id INTEGER,

    product_id INTEGER,

    event_id INTEGER,

    final_risk_score REAL NOT NULL,

    risk_band TEXT NOT NULL,

    data_coverage_score REAL,

    confidence_band TEXT,

    risk_reasons TEXT,

    risk_engine_version TEXT,

    scored_at TEXT NOT NULL,

    FOREIGN KEY (
        supplier_id
    )
    REFERENCES suppliers(supplier_id),

    FOREIGN KEY (
        product_id
    )
    REFERENCES products(product_id),

    FOREIGN KEY (
        event_id
    )
    REFERENCES supply_events(event_id)
);
""")


# ============================================================
# RISK SIGNALS
# ============================================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS risk_signals (

    signal_id INTEGER PRIMARY KEY AUTOINCREMENT,

    risk_id INTEGER NOT NULL,

    signal_name TEXT NOT NULL,

    signal_value REAL,

    contribution_weight REAL,

    created_at TEXT NOT NULL,

    FOREIGN KEY (
        risk_id
    )
    REFERENCES risk_scores(risk_id)
    ON DELETE CASCADE
);
""")


# ============================================================
# PIPELINE RUNS
# ============================================================

cursor.execute("""
CREATE TABLE IF NOT EXISTS pipeline_runs (

    run_id INTEGER PRIMARY KEY AUTOINCREMENT,

    pipeline_name TEXT NOT NULL,

    pipeline_version TEXT,

    source_file TEXT,

    records_processed INTEGER,

    records_inserted INTEGER,

    status TEXT,

    started_at TEXT NOT NULL,

    completed_at TEXT,

    error_message TEXT
);
""")


connection.commit()


print("=" * 80)
print("RELATIONAL SCHEMA CREATED SUCCESSFULLY")
print("=" * 80)

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    connection
)

display(tables)

RELATIONAL SCHEMA CREATED SUCCESSFULLY


,name
0,pipeline_runs
1,products
2,risk_scores
3,risk_signals
4,sqlite_sequence
5,suppliers
6,supply_events


In [6]:
# ============================================================
# CELL 9 — PERFORMANCE INDEXES
# ============================================================

index_statements = [

    """
    CREATE INDEX IF NOT EXISTS
    idx_suppliers_name
    ON suppliers(supplier_name);
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_products_name
    ON products(product_name);
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_events_supplier
    ON supply_events(supplier_id);
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_events_product
    ON supply_events(product_id);
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_events_type
    ON supply_events(event_type);
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_risk_supplier
    ON risk_scores(supplier_id);
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_risk_product
    ON risk_scores(product_id);
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_risk_band
    ON risk_scores(risk_band);
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_risk_score
    ON risk_scores(final_risk_score);
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_signals_risk
    ON risk_signals(risk_id);
    """
]


for statement in index_statements:

    cursor.execute(
        statement
    )


connection.commit()


print("=" * 80)
print("DATABASE INDEXES CREATED")
print("=" * 80)

DATABASE INDEXES CREATED


In [7]:
# ============================================================
# CELL 10 — SOURCE COLUMN RESOLUTION
# ============================================================

def resolve_column(
    dataframe,
    candidates
):

    normalized_columns = {
        re.sub(
            r"[^a-z0-9]",
            "",
            str(column).lower()
        ): column
        for column in dataframe.columns
    }

    for candidate in candidates:

        normalized_candidate = re.sub(
            r"[^a-z0-9]",
            "",
            candidate.lower()
        )

        if normalized_candidate in normalized_columns:

            return normalized_columns[
                normalized_candidate
            ]

    return None


COLUMN_MAP = {

    "supplier": resolve_column(
        df,
        [
            "supplier",
            "supplier_name",
            "vendor",
            "seller"
        ]
    ),

    "company": resolve_column(
        df,
        [
            "company",
            "company_name",
            "manufacturer",
            "brand"
        ]
    ),

    "product": resolve_column(
        df,
        [
            "product",
            "product_name",
            "title",
            "item"
        ]
    ),

    "category": resolve_column(
        df,
        [
            "category",
            "product_category",
            "industry"
        ]
    ),

    "source": resolve_column(
        df,
        [
            "source",
            "platform",
            "website"
        ]
    ),

    "location": resolve_column(
        df,
        [
            "location",
            "country",
            "region",
            "city"
        ]
    ),

    "event": resolve_column(
        df,
        [
            "event",
            "event_type",
            "disruption_type"
        ]
    ),

    "title": resolve_column(
        df,
        [
            "title",
            "headline",
            "event_title"
        ]
    ),

    "description": resolve_column(
        df,
        [
            "description",
            "content",
            "text",
            "review"
        ]
    ),

    "date": resolve_column(
        df,
        [
            "date",
            "event_date",
            "published_date",
            "timestamp"
        ]
    ),

    "risk_score": resolve_column(
        df,
        [
            "final_risk_score"
        ]
    ),

    "risk_band": resolve_column(
        df,
        [
            "risk_band"
        ]
    ),

    "coverage": resolve_column(
        df,
        [
            "data_coverage_score"
        ]
    ),

    "confidence": resolve_column(
        df,
        [
            "confidence_band"
        ]
    ),

    "reasons": resolve_column(
        df,
        [
            "risk_reasons_text",
            "risk_reasons"
        ]
    ),

    "engine_version": resolve_column(
        df,
        [
            "risk_engine_version"
        ]
    )
}


print("=" * 80)
print("SOURCE COLUMN MAPPING")
print("=" * 80)

for logical_name, actual_column in COLUMN_MAP.items():

    print(
        f"{logical_name:<18} : "
        f"{actual_column if actual_column else 'NOT FOUND'}"
    )

print("=" * 80)

SOURCE COLUMN MAPPING
supplier           : supplier
company            : company
product            : product
category           : NOT FOUND
source             : source
location           : location
event              : event
title              : title
description        : review
date               : NOT FOUND
risk_score         : final_risk_score
risk_band          : risk_band
coverage           : data_coverage_score
confidence         : confidence_band
reasons            : risk_reasons_text
engine_version     : risk_engine_version


In [8]:
# ============================================================
# CELL 11 — NORMALIZE DATABASE INPUT DATA
# ============================================================

db_df = df.copy()


def clean_text(value):

    if value is None:
        return None

    if isinstance(
        value,
        (list, tuple)
    ):

        return " | ".join(
            str(item)
            for item in value
        )

    if isinstance(value, dict):

        return json.dumps(
            value,
            ensure_ascii=False
        )

    if pd.isna(value):

        return None

    text = str(value).strip()

    return text if text else None


# Clean object/string columns
for column in db_df.columns:

    if (
        db_df[column].dtype == "object"
        or str(db_df[column].dtype).startswith("string")
    ):

        db_df[column] = (
            db_df[column]
            .apply(clean_text)
        )


# Risk score
if COLUMN_MAP["risk_score"]:

    db_df[
        COLUMN_MAP["risk_score"]
    ] = pd.to_numeric(
        db_df[
            COLUMN_MAP["risk_score"]
        ],
        errors="coerce"
    )


# Coverage
if COLUMN_MAP["coverage"]:

    db_df[
        COLUMN_MAP["coverage"]
    ] = pd.to_numeric(
        db_df[
            COLUMN_MAP["coverage"]
        ],
        errors="coerce"
    )


print("=" * 80)
print("DATABASE INPUT NORMALIZATION COMPLETE")
print("=" * 80)

print(
    f"Records ready for ingestion: "
    f"{len(db_df):,}"
)

DATABASE INPUT NORMALIZATION COMPLETE
Records ready for ingestion: 303


## 3. Populate Core Entity Tables

Suppliers and products are stored once and referenced through foreign keys.
This prevents unnecessary duplication and makes future WebShield queries
much more efficient.

In [9]:
# ============================================================
# CELL 13 — INSERT SUPPLIERS
# ============================================================

supplier_column = COLUMN_MAP["supplier"]
company_column = COLUMN_MAP["company"]
source_column = COLUMN_MAP["source"]
location_column = COLUMN_MAP["location"]


supplier_records = []


if supplier_column:

    supplier_working = db_df[
        [
            supplier_column,
            company_column,
            source_column,
            location_column
        ]
        if all([
            supplier_column,
            company_column,
            source_column,
            location_column
        ])
        else [
            column
            for column in [
                supplier_column,
                company_column,
                source_column,
                location_column
            ]
            if column
        ]
    ].copy()


    supplier_working = (
        supplier_working
        .drop_duplicates()
    )


    for _, row in supplier_working.iterrows():

        supplier_name = clean_text(
            row[supplier_column]
        )

        if not supplier_name:

            continue

        company_name = (
            clean_text(
                row[company_column]
            )
            if company_column
            else None
        )

        source = (
            clean_text(
                row[source_column]
            )
            if source_column
            else None
        )

        location = (
            clean_text(
                row[location_column]
            )
            if location_column
            else None
        )

        supplier_records.append(
            (
                supplier_name,
                company_name,
                source,
                location,
                datetime.now(
                    timezone.utc
                ).isoformat()
            )
        )


    cursor.executemany(
        """
        INSERT OR IGNORE INTO suppliers (
            supplier_name,
            company_name,
            source,
            location,
            created_at
        )
        VALUES (?, ?, ?, ?, ?)
        """,
        supplier_records
    )


    connection.commit()


print("=" * 80)
print("SUPPLIER TABLE POPULATED")
print("=" * 80)

supplier_count = cursor.execute(
    "SELECT COUNT(*) FROM suppliers"
).fetchone()[0]

print(
    f"Suppliers in database: "
    f"{supplier_count:,}"
)

SUPPLIER TABLE POPULATED
Suppliers in database: 0


In [10]:
# ============================================================
# CELL 14 — INSERT PRODUCTS
# ============================================================

product_column = COLUMN_MAP["product"]
category_column = COLUMN_MAP["category"]


product_records = []


if product_column:

    selected_columns = [
        column
        for column in [
            product_column,
            company_column,
            category_column,
            source_column
        ]
        if column
    ]


    product_working = (
        db_df[selected_columns]
        .drop_duplicates()
    )


    for _, row in product_working.iterrows():

        product_name = clean_text(
            row[product_column]
        )

        if not product_name:

            continue

        company_name = (
            clean_text(
                row[company_column]
            )
            if company_column
            else None
        )

        category = (
            clean_text(
                row[category_column]
            )
            if category_column
            else None
        )

        source = (
            clean_text(
                row[source_column]
            )
            if source_column
            else None
        )

        product_records.append(
            (
                product_name,
                company_name,
                category,
                source,
                datetime.now(
                    timezone.utc
                ).isoformat()
            )
        )


    cursor.executemany(
        """
        INSERT OR IGNORE INTO products (
            product_name,
            company_name,
            category,
            source,
            created_at
        )
        VALUES (?, ?, ?, ?, ?)
        """,
        product_records
    )


    connection.commit()


print("=" * 80)
print("PRODUCT TABLE POPULATED")
print("=" * 80)

product_count = cursor.execute(
    "SELECT COUNT(*) FROM products"
).fetchone()[0]

print(
    f"Products in database: "
    f"{product_count:,}"
)

PRODUCT TABLE POPULATED
Products in database: 303


In [11]:
# ============================================================
# CELL 15 — INSERT SUPPLY EVENTS
# ============================================================

import hashlib


event_column = COLUMN_MAP["event"]
title_column = COLUMN_MAP["title"]
description_column = COLUMN_MAP["description"]
date_column = COLUMN_MAP["date"]


event_records = []


def generate_record_hash(
    supplier,
    product,
    event,
    title,
    source,
    date
):

    raw = "|".join([
        str(supplier or ""),
        str(product or ""),
        str(event or ""),
        str(title or ""),
        str(source or ""),
        str(date or "")
    ])

    return hashlib.sha256(
        raw.encode("utf-8")
    ).hexdigest()


for _, row in db_df.iterrows():

    supplier = (
        clean_text(
            row[supplier_column]
        )
        if supplier_column
        else None
    )

    product = (
        clean_text(
            row[product_column]
        )
        if product_column
        else None
    )

    event = (
        clean_text(
            row[event_column]
        )
        if event_column
        else None
    )

    title = (
        clean_text(
            row[title_column]
        )
        if title_column
        else None
    )

    description = (
        clean_text(
            row[description_column]
        )
        if description_column
        else None
    )

    location = (
        clean_text(
            row[location_column]
        )
        if location_column
        else None
    )

    source = (
        clean_text(
            row[source_column]
        )
        if source_column
        else None
    )

    event_date = (
        clean_text(
            row[date_column]
        )
        if date_column
        else None
    )


    record_hash = generate_record_hash(
        supplier,
        product,
        event,
        title,
        source,
        event_date
    )


    event_records.append(
        (
            supplier,
            product,
            event,
            title,
            description,
            location,
            source,
            event_date,
            record_hash,
            datetime.now(
                timezone.utc
            ).isoformat()
        )
    )


# ------------------------------------------------------------
# Insert using lookup subqueries
# ------------------------------------------------------------

for record in event_records:

    (
        supplier,
        product,
        event,
        title,
        description,
        location,
        source,
        event_date,
        record_hash,
        created_at
    ) = record


    supplier_id = None

    if supplier:

        result = cursor.execute(
            """
            SELECT supplier_id
            FROM suppliers
            WHERE supplier_name = ?
            LIMIT 1
            """,
            (supplier,)
        ).fetchone()

        if result:
            supplier_id = result[0]


    product_id = None

    if product:

        result = cursor.execute(
            """
            SELECT product_id
            FROM products
            WHERE product_name = ?
            LIMIT 1
            """,
            (product,)
        ).fetchone()

        if result:
            product_id = result[0]


    cursor.execute(
        """
        INSERT OR IGNORE INTO supply_events (
            supplier_id,
            product_id,
            event_type,
            event_title,
            event_description,
            location,
            source,
            event_date,
            record_hash,
            created_at
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            supplier_id,
            product_id,
            event,
            title,
            description,
            location,
            source,
            event_date,
            record_hash,
            created_at
        )
    )


connection.commit()


event_count = cursor.execute(
    "SELECT COUNT(*) FROM supply_events"
).fetchone()[0]


print("=" * 80)
print("SUPPLY EVENT TABLE POPULATED")
print("=" * 80)

print(
    f"Supply events in database: "
    f"{event_count:,}"
)

SUPPLY EVENT TABLE POPULATED
Supply events in database: 302


## 4. Store Final Risk Intelligence

Risk scores are stored separately from suppliers, products, and events.
This allows the system to retain the scoring history and later support
WebShield alerts and time-based risk monitoring.

In [12]:
# ============================================================
# CELL 17 — INSERT RISK SCORES
# ============================================================

risk_score_column = COLUMN_MAP["risk_score"]
risk_band_column = COLUMN_MAP["risk_band"]
coverage_column = COLUMN_MAP["coverage"]
confidence_column = COLUMN_MAP["confidence"]
reasons_column = COLUMN_MAP["reasons"]
engine_version_column = COLUMN_MAP["engine_version"]


if not risk_score_column:

    raise ValueError(
        "final_risk_score column was not found."
    )


risk_records_inserted = 0


for _, row in db_df.iterrows():

    risk_score = pd.to_numeric(
        row[risk_score_column],
        errors="coerce"
    )

    if pd.isna(risk_score):
        continue


    risk_score = float(
        np.clip(
            risk_score,
            0,
            100
        )
    )


    risk_band = (
        clean_text(
            row[risk_band_column]
        )
        if risk_band_column
        else None
    )


    coverage = (
        pd.to_numeric(
            row[coverage_column],
            errors="coerce"
        )
        if coverage_column
        else None
    )


    if pd.isna(coverage):
        coverage = None
    else:
        coverage = float(
            np.clip(
                coverage,
                0,
                100
            )
        )


    confidence = (
        clean_text(
            row[confidence_column]
        )
        if confidence_column
        else None
    )


    reasons = (
        clean_text(
            row[reasons_column]
        )
        if reasons_column
        else None
    )


    engine_version = (
        clean_text(
            row[engine_version_column]
        )
        if engine_version_column
        else "1.0.0"
    )


    supplier = (
        clean_text(
            row[supplier_column]
        )
        if supplier_column
        else None
    )


    product = (
        clean_text(
            row[product_column]
        )
        if product_column
        else None
    )


    supplier_id = None

    if supplier:

        result = cursor.execute(
            """
            SELECT supplier_id
            FROM suppliers
            WHERE supplier_name = ?
            LIMIT 1
            """,
            (supplier,)
        ).fetchone()

        if result:
            supplier_id = result[0]


    product_id = None

    if product:

        result = cursor.execute(
            """
            SELECT product_id
            FROM products
            WHERE product_name = ?
            LIMIT 1
            """,
            (product,)
        ).fetchone()

        if result:
            product_id = result[0]


    # Try to connect the risk score to the corresponding
    # supply event using the record context.

    event_id = None

    if supplier_id or product_id:

        query = """
        SELECT event_id
        FROM supply_events
        WHERE 1 = 1
        """

        parameters = []

        if supplier_id:

            query += """
            AND supplier_id = ?
            """

            parameters.append(
                supplier_id
            )

        if product_id:

            query += """
            AND product_id = ?
            """

            parameters.append(
                product_id
            )

        query += """
        ORDER BY event_id DESC
        LIMIT 1
        """

        result = cursor.execute(
            query,
            parameters
        ).fetchone()

        if result:
            event_id = result[0]


    cursor.execute(
        """
        INSERT INTO risk_scores (
            supplier_id,
            product_id,
            event_id,
            final_risk_score,
            risk_band,
            data_coverage_score,
            confidence_band,
            risk_reasons,
            risk_engine_version,
            scored_at
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            supplier_id,
            product_id,
            event_id,
            risk_score,
            risk_band,
            coverage,
            confidence,
            reasons,
            engine_version,
            datetime.now(
                timezone.utc
            ).isoformat()
        )
    )


    risk_records_inserted += 1


connection.commit()


print("=" * 80)
print("RISK SCORES INSERTED")
print("=" * 80)

print(
    f"Risk records inserted: "
    f"{risk_records_inserted:,}"
)

RISK SCORES INSERTED
Risk records inserted: 303


In [13]:
# ============================================================
# CELL 18 — INSERT RISK SIGNALS
# ============================================================

# Candidate component signals produced by Notebook 05.

signal_candidates = {

    "NLP Risk": "risk_nlp_risk",

    "Supplier Risk": "risk_supplier_risk",

    "Quality Risk": "risk_quality_risk",

    "Counterfeit Risk": "risk_counterfeit_risk",

    "Price Pressure": "risk_price_pressure",

    "Urgency": "risk_urgency",

    "Anomaly Risk": "risk_anomaly_risk",

    "Market Risk": "risk_market_risk",

    "Disruption Risk": "risk_disruption_risk",

    "WebShield Risk": "risk_webshield_risk"
}


available_signal_candidates = {

    name: column

    for name, column
    in signal_candidates.items()

    if column in db_df.columns
}


# We use the risk record order created above.
risk_ids = [
    row[0]
    for row in cursor.execute(
        """
        SELECT risk_id
        FROM risk_scores
        ORDER BY risk_id ASC
        """
    ).fetchall()
]


signal_inserted = 0


for dataframe_index, risk_id in zip(
    db_df.index,
    risk_ids
):

    row = db_df.loc[
        dataframe_index
    ]


    for signal_name, source_column in (
        available_signal_candidates.items()
    ):

        value = pd.to_numeric(
            row[source_column],
            errors="coerce"
        )


        if pd.isna(value):
            continue


        # The scoring weights are stored for transparency.
        weight = 0.0


        weight_mapping = {

            "risk_nlp_risk": 0.20,

            "risk_supplier_risk": 0.15,

            "risk_quality_risk": 0.10,

            "risk_counterfeit_risk": 0.15,

            "risk_price_pressure": 0.10,

            "risk_urgency": 0.05,

            "risk_anomaly_risk": 0.10,

            "risk_market_risk": 0.05,

            "risk_disruption_risk": 0.05,

            "risk_webshield_risk": 0.05
        }


        weight = weight_mapping.get(
            source_column,
            0.0
        )


        cursor.execute(
            """
            INSERT INTO risk_signals (
                risk_id,
                signal_name,
                signal_value,
                contribution_weight,
                created_at
            )
            VALUES (?, ?, ?, ?, ?)
            """,
            (
                risk_id,
                signal_name,
                float(
                    np.clip(
                        value,
                        0,
                        100
                    )
                ),
                weight,
                datetime.now(
                    timezone.utc
                ).isoformat()
            )
        )


        signal_inserted += 1


connection.commit()


print("=" * 80)
print("RISK SIGNALS INSERTED")
print("=" * 80)

print(
    f"Signal records inserted: "
    f"{signal_inserted:,}"
)

RISK SIGNALS INSERTED
Signal records inserted: 3,030


In [14]:
# ============================================================
# CELL 19 — PIPELINE RUN AUDIT
# ============================================================

pipeline_started = datetime.now(
    timezone.utc
).isoformat()


cursor.execute(
    """
    INSERT INTO pipeline_runs (
        pipeline_name,
        pipeline_version,
        source_file,
        records_processed,
        records_inserted,
        status,
        started_at,
        completed_at,
        error_message
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """,
    (
        "SupplyShield SQL Integration",
        "1.0.0",
        str(input_source),
        len(df),
        risk_records_inserted,
        "SUCCESS",
        pipeline_started,
        datetime.now(
            timezone.utc
        ).isoformat(),
        None
    )
)


connection.commit()


print("=" * 80)
print("PIPELINE RUN LOGGED")
print("=" * 80)

PIPELINE RUN LOGGED


## 5. SQL Analytics Layer

The following queries demonstrate how SupplyShield can retrieve actionable
intelligence directly from SQL rather than recalculating risk in the UI.

In [15]:
# ============================================================
# CELL 21 — DATABASE OVERVIEW
# ============================================================

tables_df = pd.read_sql_query(
    """
    SELECT
        name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    connection
)


print("=" * 80)
print("DATABASE TABLE OVERVIEW")
print("=" * 80)


for table_name in tables_df["table_name"]:

    count = cursor.execute(
        f"""
        SELECT COUNT(*)
        FROM "{table_name}"
        """
    ).fetchone()[0]

    print(
        f"{table_name:<20} : "
        f"{count:,} records"
    )


print("=" * 80)

DATABASE TABLE OVERVIEW
pipeline_runs        : 1 records
products             : 303 records
risk_scores          : 303 records
risk_signals         : 3,030 records
sqlite_sequence      : 5 records
suppliers            : 0 records
supply_events        : 302 records


In [16]:
# ============================================================
# CELL 22 — SQL QUERY: TOP HIGH-RISK RECORDS
# ============================================================

query_top_risk = """
SELECT

    rs.risk_id,

    s.supplier_name,

    p.product_name,

    rs.final_risk_score,

    rs.risk_band,

    rs.data_coverage_score,

    rs.confidence_band,

    rs.risk_reasons,

    rs.scored_at

FROM risk_scores rs

LEFT JOIN suppliers s
    ON rs.supplier_id = s.supplier_id

LEFT JOIN products p
    ON rs.product_id = p.product_id

ORDER BY
    rs.final_risk_score DESC

LIMIT 20;
"""


top_risk_sql = pd.read_sql_query(
    query_top_risk,
    connection
)


display(top_risk_sql)

,risk_id,supplier_name,product_name,final_risk_score,risk_band,data_coverage_score,confidence_band,risk_reasons,scored_at
0,6,None,Leak Proof Tape – Instant Waterproof Seal for Repairs,11.45,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (98.7/100),2026-08-21T04:24:41.676043+00:00
1,100,None,Siemens Artis Zee Cath Lab,11.13,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (99.0/100),2026-08-21T04:24:41.714764+00:00
2,128,None,Allenger Altima F100 Fixed Cath Lab Machine,11.00,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (100.0/100),2026-08-21T04:24:41.720531+00:00
3,239,None,Cath Lab Machine,10.67,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (96.7/100),2026-08-21T04:24:41.764966+00:00
4,168,None,Broken Bag Detector,9.88,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (81.5/100),2026-08-21T04:24:41.736407+00:00
5,12,None,Manual Wall Fastening Nail Gun Tool for Wood and Concrete Walls (1 Set),9.70,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (81.1/100),2026-08-21T04:24:41.679593+00:00
6,3,None,Manual Wall Fastening Nail Gun Tool Set (1 Set),9.66,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (80.7/100),2026-08-21T04:24:41.674537+00:00
7,43,None,Broken Bag Detector,9.62,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (78.3/100),2026-08-21T04:24:41.690604+00:00
8,201,None,Broken Bag Detector,9.49,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (77.6/100),2026-08-21T04:24:41.748576+00:00
9,9,None,Hardware Tool Set – 11 Pcs Multi-Functional Kit,9.48,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (84.8/100),2026-08-21T04:24:41.677672+00:00


In [17]:
# ============================================================
# CELL 23 — SQL QUERY: SUPPLIER RISK ANALYTICS
# ============================================================

query_supplier_risk = """
SELECT

    s.supplier_name,

    COUNT(rs.risk_id)
        AS risk_records,

    ROUND(
        AVG(rs.final_risk_score),
        2
    )
        AS average_risk,

    ROUND(
        MAX(rs.final_risk_score),
        2
    )
        AS maximum_risk,

    SUM(
        CASE
            WHEN rs.risk_band = 'CRITICAL'
            THEN 1
            ELSE 0
        END
    )
        AS critical_count,

    SUM(
        CASE
            WHEN rs.risk_band = 'HIGH'
            THEN 1
            ELSE 0
        END
    )
        AS high_count

FROM suppliers s

LEFT JOIN risk_scores rs
    ON s.supplier_id = rs.supplier_id

GROUP BY
    s.supplier_id,
    s.supplier_name

ORDER BY
    average_risk DESC;
"""


supplier_sql = pd.read_sql_query(
    query_supplier_risk,
    connection
)


display(
    supplier_sql.head(25)
)

,supplier_name,risk_records,average_risk,maximum_risk,critical_count,high_count


In [18]:
# ============================================================
# CELL 24 — SQL QUERY: RISK DISTRIBUTION
# ============================================================

query_distribution = """
SELECT

    risk_band,

    COUNT(*) AS record_count,

    ROUND(
        AVG(final_risk_score),
        2
    ) AS average_score,

    ROUND(
        MIN(final_risk_score),
        2
    ) AS minimum_score,

    ROUND(
        MAX(final_risk_score),
        2
    ) AS maximum_score

FROM risk_scores

GROUP BY
    risk_band

ORDER BY
    average_score DESC;
"""


risk_distribution_sql = pd.read_sql_query(
    query_distribution,
    connection
)


display(
    risk_distribution_sql
)

,risk_band,record_count,average_score,minimum_score,maximum_score
0,LOW,303,3.0,0.24,11.45


In [19]:
# ============================================================
# CELL 25 — SQL QUERY: RISK SIGNAL ANALYTICS
# ============================================================

query_signals = """
SELECT

    signal_name,

    COUNT(*) AS observations,

    ROUND(
        AVG(signal_value),
        2
    ) AS average_signal,

    ROUND(
        MAX(signal_value),
        2
    ) AS maximum_signal,

    ROUND(
        AVG(contribution_weight) * 100,
        2
    ) AS configured_weight_percentage

FROM risk_signals

GROUP BY
    signal_name

ORDER BY
    average_signal DESC;
"""


signal_sql = pd.read_sql_query(
    query_signals,
    connection
)


display(signal_sql)

,signal_name,observations,average_signal,maximum_signal,configured_weight_percentage
0,Anomaly Risk,303,20.17,100.00,10.0
1,NLP Risk,303,4.73,9.79,20.0
2,Price Pressure,303,0.33,50.00,10.0
3,WebShield Risk,303,0.00,0.00,5.0
4,Urgency,303,0.00,0.00,5.0
5,Supplier Risk,303,0.00,0.00,15.0
6,Quality Risk,303,0.00,0.00,10.0
7,Market Risk,303,0.00,0.00,5.0
8,Disruption Risk,303,0.00,0.00,5.0
9,Counterfeit Risk,303,0.00,0.00,15.0


## 6. Dashboard-Ready SQL Views

Views provide stable query interfaces for the Streamlit application and the
future WebShield layer.

In [20]:
# ============================================================
# CELL 27 — CREATE DASHBOARD-READY SQL VIEWS
# ============================================================


# ------------------------------------------------------------
# Main risk dashboard view
# ------------------------------------------------------------

cursor.execute(
    """
    CREATE VIEW IF NOT EXISTS vw_risk_dashboard AS

    SELECT

        rs.risk_id,

        s.supplier_name,

        s.company_name,

        p.product_name,

        p.category,

        se.event_type,

        se.event_title,

        se.location,

        se.source,

        rs.final_risk_score,

        rs.risk_band,

        rs.data_coverage_score,

        rs.confidence_band,

        rs.risk_reasons,

        rs.risk_engine_version,

        rs.scored_at

    FROM risk_scores rs

    LEFT JOIN suppliers s
        ON rs.supplier_id = s.supplier_id

    LEFT JOIN products p
        ON rs.product_id = p.product_id

    LEFT JOIN supply_events se
        ON rs.event_id = se.event_id;
    """
)


# ------------------------------------------------------------
# Supplier dashboard view
# ------------------------------------------------------------

cursor.execute(
    """
    CREATE VIEW IF NOT EXISTS vw_supplier_risk AS

    SELECT

        s.supplier_id,

        s.supplier_name,

        s.company_name,

        s.location,

        COUNT(rs.risk_id)
            AS risk_records,

        ROUND(
            AVG(rs.final_risk_score),
            2
        )
            AS average_risk,

        ROUND(
            MAX(rs.final_risk_score),
            2
        )
            AS maximum_risk,

        ROUND(
            AVG(rs.data_coverage_score),
            2
        )
            AS average_data_coverage,

        SUM(
            CASE
                WHEN rs.risk_band = 'CRITICAL'
                THEN 1
                ELSE 0
            END
        )
            AS critical_alerts,

        SUM(
            CASE
                WHEN rs.risk_band = 'HIGH'
                THEN 1
                ELSE 0
            END
        )
            AS high_alerts

    FROM suppliers s

    LEFT JOIN risk_scores rs
        ON s.supplier_id = rs.supplier_id

    GROUP BY
        s.supplier_id,
        s.supplier_name,
        s.company_name,
        s.location;
    """
)


# ------------------------------------------------------------
# Risk signal dashboard view
# ------------------------------------------------------------

cursor.execute(
    """
    CREATE VIEW IF NOT EXISTS vw_risk_signals AS

    SELECT

        rs.risk_id,

        rs.final_risk_score,

        rs.risk_band,

        rsg.signal_name,

        rsg.signal_value,

        rsg.contribution_weight

    FROM risk_scores rs

    INNER JOIN risk_signals rsg

        ON rs.risk_id = rsg.risk_id;
    """
)


connection.commit()


print("=" * 80)
print("DASHBOARD VIEWS CREATED")
print("=" * 80)

views = pd.read_sql_query(
    """
    SELECT
        name
    FROM sqlite_master
    WHERE type = 'view'
    ORDER BY name;
    """,
    connection
)

display(views)

DASHBOARD VIEWS CREATED


,name
0,vw_risk_dashboard
1,vw_risk_signals
2,vw_supplier_risk


In [21]:
# ============================================================
# CELL 28 — TEST DASHBOARD VIEWS
# ============================================================

dashboard_test = pd.read_sql_query(
    """
    SELECT *
    FROM vw_risk_dashboard
    ORDER BY final_risk_score DESC
    LIMIT 10;
    """,
    connection
)


supplier_view_test = pd.read_sql_query(
    """
    SELECT *
    FROM vw_supplier_risk
    ORDER BY average_risk DESC
    LIMIT 10;
    """,
    connection
)


signal_view_test = pd.read_sql_query(
    """
    SELECT *
    FROM vw_risk_signals
    ORDER BY signal_value DESC
    LIMIT 10;
    """,
    connection
)


print("=" * 80)
print("RISK DASHBOARD VIEW")
print("=" * 80)

display(
    dashboard_test
)


print("=" * 80)
print("SUPPLIER RISK VIEW")
print("=" * 80)

display(
    supplier_view_test
)


print("=" * 80)
print("RISK SIGNAL VIEW")
print("=" * 80)

display(
    signal_view_test
)

RISK DASHBOARD VIEW


,risk_id,supplier_name,company_name,product_name,category,event_type,event_title,location,source,final_risk_score,risk_band,data_coverage_score,confidence_band,risk_reasons,risk_engine_version,scored_at
0,6,None,None,Leak Proof Tape – Instant Waterproof Seal for Repairs,None,None,Leak Proof Tape – Instant Waterproof Seal for Repairs | BoltForce,None,DeoDap,11.45,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (98.7/100),1.0.0,2026-08-21T04:24:41.676043+00:00
1,100,None,None,Siemens Artis Zee Cath Lab,None,None,Siemens Artis Zee Cath Lab - Application: Hospital,None,TradeIndia,11.13,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (99.0/100),1.0.0,2026-08-21T04:24:41.714764+00:00
2,128,None,None,Allenger Altima F100 Fixed Cath Lab Machine,None,None,Allenger Altima F100 Fixed Cath Lab Machine,None,TradeIndia,11.00,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (100.0/100),1.0.0,2026-08-21T04:24:41.720531+00:00
3,239,None,None,Cath Lab Machine,None,None,Cath Lab Machine,None,TradeIndia,10.67,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (96.7/100),1.0.0,2026-08-21T04:24:41.764966+00:00
4,168,None,None,Broken Bag Detector,None,None,"Broken Bag Detector - 315 Grade Stainless Steel, 115x65x55 Mm | Adjustable Sensitivity, Instanta...",None,TradeIndia,9.88,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (81.5/100),1.0.0,2026-08-21T04:24:41.736407+00:00
5,12,None,None,Manual Wall Fastening Nail Gun Tool for Wood and Concrete Walls (1 Set),None,None,Manual Wall Fastening Nail Gun Tool for Wood and Concrete Walls (1 Set),None,DeoDap,9.70,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (81.1/100),1.0.0,2026-08-21T04:24:41.679593+00:00
6,3,None,None,Manual Wall Fastening Nail Gun Tool Set (1 Set),None,None,Manual Wall Fastening Nail Gun Tool Set (1 Set),None,DeoDap,9.66,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (80.7/100),1.0.0,2026-08-21T04:24:41.674537+00:00
7,43,None,None,Broken Bag Detector,None,None,"Broken Bag Detector - 315 Grade Stainless Steel, 115x65x55 Mm | Adjustable Sensitivity, Instanta...",None,TradeIndia,9.62,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (78.3/100),1.0.0,2026-08-21T04:24:41.690604+00:00
8,201,None,None,Broken Bag Detector,None,None,"Broken Bag Detector - 315 Grade Stainless Steel, 115x65x55 Mm | Adjustable Sensitivity, Instanta...",None,TradeIndia,9.49,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (77.6/100),1.0.0,2026-08-21T04:24:41.748576+00:00
9,9,None,None,Hardware Tool Set – 11 Pcs Multi-Functional Kit,None,None,Hardware Tool Set – 11 Pcs Multi-Functional Kit | BoltForce,None,DeoDap,9.48,LOW,100.0,HIGH,Unusual/anomalous behaviour detected (84.8/100),1.0.0,2026-08-21T04:24:41.677672+00:00


SUPPLIER RISK VIEW


,supplier_id,supplier_name,company_name,location,risk_records,average_risk,maximum_risk,average_data_coverage,critical_alerts,high_alerts


RISK SIGNAL VIEW


,risk_id,final_risk_score,risk_band,signal_name,signal_value,contribution_weight
0,128,11.00,LOW,Anomaly Risk,100.000000,0.1
1,100,11.13,LOW,Anomaly Risk,99.041098,0.1
2,6,11.45,LOW,Anomaly Risk,98.687571,0.1
3,239,10.67,LOW,Anomaly Risk,96.714482,0.1
4,5,8.82,LOW,Anomaly Risk,85.001131,0.1
5,9,9.48,LOW,Anomaly Risk,84.803158,0.1
6,168,9.88,LOW,Anomaly Risk,81.485541,0.1
7,12,9.70,LOW,Anomaly Risk,81.111202,0.1
8,3,9.66,LOW,Anomaly Risk,80.741588,0.1
9,43,9.62,LOW,Anomaly Risk,78.289632,0.1


## 7. Database Integrity Validation

Before the database is used by the WebShield and dashboard layers, the
relational structure is checked for orphaned records, invalid scores,
missing critical fields, and foreign-key violations.

In [22]:
# ============================================================
# CELL 30 — DATABASE INTEGRITY VALIDATION
# ============================================================

validation = {}


# ------------------------------------------------------------
# SQLite foreign key validation
# ------------------------------------------------------------

foreign_key_violations = pd.read_sql_query(
    """
    PRAGMA foreign_key_check;
    """,
    connection
)

validation[
    "foreign_key_integrity"
] = (
    len(foreign_key_violations) == 0
)


# ------------------------------------------------------------
# Risk score range
# ------------------------------------------------------------

invalid_scores = cursor.execute(
    """
    SELECT COUNT(*)
    FROM risk_scores
    WHERE final_risk_score < 0
       OR final_risk_score > 100;
    """
).fetchone()[0]


validation[
    "risk_score_range"
] = (
    invalid_scores == 0
)


# ------------------------------------------------------------
# Invalid risk bands
# ------------------------------------------------------------

invalid_bands = cursor.execute(
    """
    SELECT COUNT(*)
    FROM risk_scores
    WHERE risk_band NOT IN (
        'LOW',
        'MEDIUM',
        'HIGH',
        'CRITICAL'
    );
    """
).fetchone()[0]


validation[
    "risk_band_values"
] = (
    invalid_bands == 0
)


# ------------------------------------------------------------
# Risk records
# ------------------------------------------------------------

risk_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM risk_scores;
    """
).fetchone()[0]


validation[
    "risk_records_exist"
] = (
    risk_count > 0
)


# ------------------------------------------------------------
# Pipeline status
# ------------------------------------------------------------

pipeline_status = cursor.execute(
    """
    SELECT status
    FROM pipeline_runs
    ORDER BY run_id DESC
    LIMIT 1;
    """
).fetchone()


validation[
    "pipeline_success"
] = (
    pipeline_status is not None
    and pipeline_status[0] == "SUCCESS"
)


# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("=" * 80)
print("DATABASE INTEGRITY VALIDATION")
print("=" * 80)

all_passed = True

for check, result in validation.items():

    status = "PASS" if result else "FAIL"

    print(
        f"{check:<30} : {status}"
    )

    if not result:
        all_passed = False


print("-" * 80)

if all_passed:

    print(
        "ALL DATABASE VALIDATION CHECKS PASSED."
    )

else:

    print(
        "WARNING: DATABASE VALIDATION REQUIRES ATTENTION."
    )

print("=" * 80)

DATABASE INTEGRITY VALIDATION
foreign_key_integrity          : PASS
risk_score_range               : PASS
risk_band_values               : PASS
risk_records_exist             : PASS
pipeline_success               : PASS
--------------------------------------------------------------------------------
ALL DATABASE VALIDATION CHECKS PASSED.


In [23]:
# ============================================================
# CELL 31 — FINAL DATABASE STATISTICS
# ============================================================

database_statistics = {}


for table_name in [
    "suppliers",
    "products",
    "supply_events",
    "risk_scores",
    "risk_signals",
    "pipeline_runs"
]:

    count = cursor.execute(
        f"""
        SELECT COUNT(*)
        FROM {table_name};
        """
    ).fetchone()[0]

    database_statistics[
        table_name
    ] = count


print("=" * 80)
print("SUPPLYSHIELD DATABASE STATISTICS")
print("=" * 80)

for table_name, count in database_statistics.items():

    print(
        f"{table_name:<20} : "
        f"{count:,} records"
    )


database_size_mb = (
    DATABASE_PATH.stat().st_size
    / (1024 ** 2)
)


print("-" * 80)

print(
    f"Database size        : "
    f"{database_size_mb:.2f} MB"
)

print(
    f"Database location    : "
    f"{DATABASE_PATH}"
)

print("=" * 80)

SUPPLYSHIELD DATABASE STATISTICS
suppliers            : 0 records
products             : 303 records
supply_events        : 302 records
risk_scores          : 303 records
risk_signals         : 3,030 records
pipeline_runs        : 1 records
--------------------------------------------------------------------------------
Database size        : 0.00 MB
Database location    : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\database\supplyshield.db


In [25]:
# ============================================================
# CELL 32 — EXPORT SQL ANALYTICS FOR REPORTING
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# Resolve project root safely
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd()

PROJECT_ROOT_EXPORT = CURRENT_DIR

for parent in [CURRENT_DIR] + list(CURRENT_DIR.parents):

    if (parent / "member2").exists():
        PROJECT_ROOT_EXPORT = parent
        break


# ------------------------------------------------------------
# Create SQL analytics output directory
# ------------------------------------------------------------

SQL_ANALYTICS_DIR = (
    PROJECT_ROOT_EXPORT
    / "member2"
    / "data"
    / "processed"
    / "sql_analytics"
)

SQL_ANALYTICS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 80)
print("EXPORTING SQL ANALYTICS")
print("=" * 80)

print(
    f"Output directory:\n{SQL_ANALYTICS_DIR}"
)


# ============================================================
# 1. DASHBOARD RISK DATA
# ============================================================

dashboard_full = pd.read_sql_query(
    """
    SELECT *
    FROM vw_risk_dashboard
    ORDER BY final_risk_score DESC;
    """,
    connection
)

dashboard_path = (
    SQL_ANALYTICS_DIR
    / "dashboard_risk_data.csv"
)

dashboard_full.to_csv(
    dashboard_path,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"✓ Dashboard risk data exported: "
    f"{len(dashboard_full):,} records"
)


# ============================================================
# 2. SUPPLIER RISK ANALYTICS
# ============================================================

supplier_full = pd.read_sql_query(
    """
    SELECT *
    FROM vw_supplier_risk
    ORDER BY average_risk DESC;
    """,
    connection
)

supplier_path = (
    SQL_ANALYTICS_DIR
    / "supplier_risk_analytics.csv"
)

supplier_full.to_csv(
    supplier_path,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"✓ Supplier analytics exported: "
    f"{len(supplier_full):,} records"
)


# ============================================================
# 3. RISK SIGNAL ANALYTICS
# ============================================================

signals_full = pd.read_sql_query(
    """
    SELECT *
    FROM vw_risk_signals
    ORDER BY signal_value DESC;
    """,
    connection
)

signals_path = (
    SQL_ANALYTICS_DIR
    / "risk_signal_analytics.csv"
)

signals_full.to_csv(
    signals_path,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"✓ Risk signal analytics exported: "
    f"{len(signals_full):,} records"
)


# ============================================================
# 4. EXPORT DATABASE METADATA
# ============================================================

metadata = pd.DataFrame({

    "metric": [
        "database_path",
        "dashboard_records",
        "supplier_records",
        "signal_records",
        "export_timestamp"
    ],

    "value": [
        str(DATABASE_PATH),
        len(dashboard_full),
        len(supplier_full),
        len(signals_full),
        datetime.now(
            timezone.utc
        ).isoformat()
    ]
})


metadata_path = (
    SQL_ANALYTICS_DIR
    / "sql_pipeline_metadata.csv"
)

metadata.to_csv(
    metadata_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# FINAL EXPORT VALIDATION
# ============================================================

export_files = [
    dashboard_path,
    supplier_path,
    signals_path,
    metadata_path
]


print()
print("=" * 80)
print("EXPORT VALIDATION")
print("=" * 80)


all_exports_valid = True


for file_path in export_files:

    exists = file_path.exists()

    size_kb = (
        file_path.stat().st_size / 1024
        if exists
        else 0
    )

    status = (
        "PASS"
        if exists and size_kb > 0
        else "FAIL"
    )

    print(
        f"{file_path.name:<35} "
        f"{status:<8} "
        f"{size_kb:>10.2f} KB"
    )

    if not exists or size_kb <= 0:

        all_exports_valid = False


print("-" * 80)

if all_exports_valid:

    print(
        "ALL SQL ANALYTICS EXPORTS COMPLETED SUCCESSFULLY."
    )

else:

    raise RuntimeError(
        "One or more SQL analytics exports failed."
    )


print("=" * 80)
print(
    f"Files saved to:\n{SQL_ANALYTICS_DIR}"
)
print("=" * 80)

EXPORTING SQL ANALYTICS
Output directory:
c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\data\processed\sql_analytics
✓ Dashboard risk data exported: 303 records
✓ Supplier analytics exported: 0 records
✓ Risk signal analytics exported: 3,030 records

EXPORT VALIDATION
dashboard_risk_data.csv             PASS          73.49 KB
supplier_risk_analytics.csv         PASS           0.14 KB
risk_signal_analytics.csv           PASS         109.03 KB
sql_pipeline_metadata.csv           PASS           0.23 KB
--------------------------------------------------------------------------------
ALL SQL ANALYTICS EXPORTS COMPLETED SUCCESSFULLY.
Files saved to:
c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\data\processed\sql_analytics


In [26]:
# ============================================================
# CELL 33 — FINAL DATABASE CHECKPOINT
# ============================================================

# Force all pending writes to disk.
connection.commit()


# Run SQLite integrity check.
integrity_result = cursor.execute(
    """
    PRAGMA integrity_check;
    """
).fetchone()[0]


print()
print("=" * 85)
print("                    SUPPLYSHIELD AI")
print("                 SQL DATABASE REPORT")
print("=" * 85)

print(
    f"Database file         : {DATABASE_PATH}"
)

print(
    f"Suppliers             : "
    f"{database_statistics['suppliers']:,}"
)

print(
    f"Products              : "
    f"{database_statistics['products']:,}"
)

print(
    f"Supply events         : "
    f"{database_statistics['supply_events']:,}"
)

print(
    f"Risk scores           : "
    f"{database_statistics['risk_scores']:,}"
)

print(
    f"Risk signals          : "
    f"{database_statistics['risk_signals']:,}"
)

print(
    f"Pipeline runs         : "
    f"{database_statistics['pipeline_runs']:,}"
)

print(
    f"SQLite integrity      : "
    f"{integrity_result}"
)

print(
    f"Validation status     : "
    f"{'PASSED' if all_passed else 'REQUIRES ATTENTION'}"
)

print("-" * 85)

print(
    "DATABASE READY FOR WEBSHIELD + DASHBOARD INTEGRATION"
)

print("=" * 85)


                    SUPPLYSHIELD AI
                 SQL DATABASE REPORT
Database file         : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\database\supplyshield.db
Suppliers             : 0
Products              : 303
Supply events         : 302
Risk scores           : 303
Risk signals          : 3,030
Pipeline runs         : 1
SQLite integrity      : ok
Validation status     : PASSED
-------------------------------------------------------------------------------------
DATABASE READY FOR WEBSHIELD + DASHBOARD INTEGRATION


In [27]:
# ============================================================
# CELL 34 — SAFE DATABASE CHECKPOINT
# ============================================================

connection.commit()

print(
    "All SQL transactions committed successfully."
)

print(
    f"Database persisted at:\n{DATABASE_PATH}"
)

# Do NOT delete the database.
# The file is now ready for Notebook 07 / WebShield / Dashboard.

All SQL transactions committed successfully.
Database persisted at:
c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\database\supplyshield.db
